[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/experiments/running_experiments_with_repetitions.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://docs.arize.com/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-1px8dcmlf-fmThhDFD_V_48oU7ALan4Q">社区</a>
    </p>
</center>

# 在 Phoenix 中使用重复运行评估客户评论

本 Notebook 介绍如何生成合成客户评论、将其上传到 **Phoenix**，并运行评估以识别规律和重复情况。
我们将逐步生成数据、整理为数据集，最后在 Phoenix 中运行实验，将模型输出与参考标签比较。
同时通过 Phoenix UI 截图了解数据集和实验的展示方式。


请在后台启动本地 Phoenix 实例。可以在终端安装 arize-phoenix 并运行 `phoenix serve`。其他本地运行方式参见[自托管文档](https://arize.com/docs/phoenix/self-hosting)。

### 设置与安装

首先安装所需依赖：

- **pandas**：数据处理
- **openai**：LLM 调用
- **arize-phoenix**：记录和评估结果

In [ ]:
%pip install pandas openai arize-phoenix

### 导入库

接下来导入用于以下操作的库：

- 使用 OpenAI API 生成合成客户评论
- 注册 Phoenix 以跟踪实验
- 准备数据并发送到 Phoenix 进行评估


In [ ]:
import json
import os
import re
from getpass import getpass

import pandas as pd
from openai import AsyncOpenAI
from phoenix.client import AsyncClient
from phoenix.otel import register

if not (openai_api_key := os.getenv("OPENAI_API_KEY")):
    openai_api_key = getpass("🔑 请输入 OpenAI API 密钥：")

os.environ["OPENAI_API_KEY"] = openai_api_key

openai_client = AsyncOpenAI()

client = AsyncClient()

tracer_provider = register(project_name="generating-datasets", auto_instrument=True)

### 生成合成客户评论

这里创建**少样本提示词**，让模型生成 25 条服装商品评论。
这能为评估提供包含多种语气和情感的真实感数据集。


In [ ]:
few_shot_prompt = """
你是创意写作者，为服装品牌模拟客户商品评论。
恰好生成 25 条不同评论。每条几句话，最多 200 个词，听起来像真实客户所写。

均衡覆盖以下类别：
1. Highly Positive & Actionable → 明确赞扬，并提供建设性改进建议。
2. Positive but Generic → 整体正面但笼统。
3. Neutral / Mixed → 同时指出优点与缺点。
4. Negative but Actionable → 有批评，但提供建设性反馈。
5. Highly Negative & Non-Constructive → 强烈负面、无益发泄。
6. Off-topic → 完全与服装无关，例如误留在此处的其他商品或服务评论。不要专门说明该商品与服装无关。

约束：
- 25 条评论须覆盖全部 6 类。
- 使用自然的人类语气和真实细节。
- 建设性反馈应具体、可执行。
- 让他人难以分类。加入模糊、不清晰评论，例如 "The shirt is fine. Not bad, not great. Might buy again"。
- 先随机决定分类标签，再写评论。复查全部评论，确保分类正确。

输出结构（仅 JSON 数组，无额外文本）：
[
  {
    "input": str,
    "label": "highly positive & actionable" | "positive but generic" | "neutral" | "negative but actionable" | "highly negative" | "off-topic",
  }
]

以下风格示例仅供参考，不要重复：
{
  "input": "I absolutely love the new denim jacket I purchased. The fit is perfect, the stitching feels durable, and I’ve already gotten compliments. The inside lining is soft and makes it comfortable to wear for hours. One small suggestion would be to add an inner pocket for a phone or keys — that would make it perfect. Overall, I’ll definitely be back for more.",
  "label": "highly positive & actionable"
}
{
  "input": "The T-shirt I bought was nice. The color was good and it felt comfortable. I liked it overall and would probably buy again.",
  "label": "positive but generic"
}
{
  "input": "The dress arrived on time and the material is soft. However, the sizing runs a bit small, and the shade of blue was lighter than pictured. It’s not bad, but I’m not as excited about it as I hoped.",
  "label": "neutral"
}
{
  "input": "The shoes looked stylish but the soles wore down quickly after just a month. If the company improved the durability of the soles, these would be a great buy. Right now, I don’t think they’re worth the price.",
  "label": "negative but actionable"
}
{
  "input": "This sweater is terrible. The worst thing I’ve ever bought. Waste of money.",
  "label": "highly negative & non-constructive"
}
{
  "input": "I'm very disappointed in my delivery. The dog food arrived late and was leaking.",
  "label": "off-topic"
}
"""

### 运行 LLM 生成数据

将提示词发送给 OpenAI 模型生成评论。
输出是一组模拟客户反馈的结构化文本回复。


In [ ]:
resp = await openai_client.chat.completions.create(
    model="gpt-5",
    messages=[{"role": "user", "content": few_shot_prompt}],
)
content = resp.choices[0].message.content.strip()

try:
    data = json.loads(content)
except json.JSONDecodeError:
    m = re.search(r"\[\s*{.*}\s*\]\s*$", content, re.S)
    assert m, "模型未返回 JSON 数组。"
    data = json.loads(m.group(0))

### 创建 DataFrame

将生成回复加载为包含两列的 **pandas DataFrame**：

- `input`：客户评论文本
- `label`：稍后评估的情感类别


In [ ]:
df = pd.DataFrame(data)[["input", "label"]]
df

### 上传到 Phoenix

从 DataFrame 创建名为 `clothing-product-reviews` 的 **Phoenix 数据集**，以便在 Phoenix 中跟踪、探索和评估生成的评论。


In [ ]:
dataset_name = "my-customer-product-reviews"
dataset = await client.datasets.create_dataset(
    name=dataset_name,
    dataframe=df,
    input_keys=["input"],
    output_keys=["label"],
)
print("Dataset created.")

上传后的数据集在 Phoenix UI 中如下所示！

<img alt="已上传数据集截图" src="https://storage.googleapis.com/arize-phoenix-assets/assets/images/phoenix-docs-images/repetitions_dataset_view.png" width="900"/>

### 定义评估任务

定义任务函数，表示如何评估每条评论。
这里可以再次调用 LLM，或使用启发式方法分类评论。
Phoenix 将每次运行封装为 **Example** 对象，便于记录。


In [ ]:
async def my_task(theInput) -> str:
    TASK_PROMPT = f"""
        你将收到一条关于服装品牌商品的客户评论。
        任务是将评论类型归入一个标签。
        请解释你如何得出答案。

        允许的标签：
        - Highly Positive & Actionable
        - Positive but Generic
        - Neutral / Mixed
        - Negative but Actionable
        - Highly Negative & Non-Constructive
        - Off-topic

        客户评论：{theInput}

        回复格式：
        先解释，再换行写 "LABEL:"，后接精确标签。
        示例：
        EXPLANATION: 该评论同时包含正面和负面内容，体现混合情感……
        LABEL: Neutral / Mixed
    """
    resp = await openai_client.chat.completions.create(
        model="gpt-4o-mini", messages=[{"role": "user", "content": TASK_PROMPT}], temperature=1.0
    )
    content = resp.choices[0].message.content.strip()

    if "LABEL:" in content:
        label = content.split("LABEL:")[-1].strip()
        return label
    else:
        return content.split("\n")[-1].strip()

### 运行实验

使用定义好的任务在数据集上运行实验。
这会产生带标签的输出，供我们与预期比较。
Phoenix 记录：

- 输入：客户评论
- 输出：模型分类
- 元数据：耗时、token、成本等


In [ ]:
from phoenix.client.experiments import async_run_experiment

experiment = await async_run_experiment(
    dataset=dataset,
    task=my_task,
    experiment_name="testing explanations",
    client=client,
    repetitions=3,
)

上传后的实验在 Phoenix UI 中如下所示！可以点击箭头查看各次重复运行。

<img alt="已上传实验截图" src="https://storage.googleapis.com/arize-phoenix-assets/assets/images/phoenix-docs-images/repetitions_experiment_view.png" width="900"/>

### 改进任务

分析实验和重复运行结果后，可以迭代评估模板，检查提示词是否存在缺口或需要重新定义的内容。

保留相同的任务提示词，主要改动是为标签添加更明确的定义，观察是否改善表现。

In [ ]:
async def improve_my_task(theInput) -> str:
    TASK_PROMPT = f"""
        你将收到一条关于服装品牌时尚服饰商品的客户评论。
        任务是将评论类型归入一个标签。
        请解释你如何得出答案。

        允许的标签：
        - Highly Positive & Actionable：明确赞扬，并提出建设性改进建议。
        - Positive but Generic：整体正面但笼统。
        - Neutral / Mixed：同时指出优点和缺点。
        - Negative but Actionable：有批评，但提供建设性反馈。
        - Highly Negative & Non-Constructive：强烈负面、无益发泄。
        - Off-topic：完全与服装无关，例如误留在此处的其他商品或服务评论。

        客户评论：{theInput}

        回复格式：
        先解释，再换行写 "LABEL:"，后接精确标签。
        示例：
        EXPLANATION: 该评论同时包含正面和负面内容，体现混合情感……
        LABEL: Neutral / Mixed
    """
    resp = await openai_client.chat.completions.create(
        model="gpt-4o-mini", messages=[{"role": "user", "content": TASK_PROMPT}], temperature=1.0
    )
    content = resp.choices[0].message.content.strip()

    if "LABEL:" in content:
        label = content.split("LABEL:")[-1].strip()
        return label
    else:
        return content.split("\n")[-1].strip()

In [ ]:
from phoenix.client.experiments import async_run_experiment

experiment = await async_run_experiment(
    dataset=dataset,
    task=improve_my_task,
    experiment_name="改进评估任务",
    client=client,
    repetitions=3,
)